# Creating the Silver Layer - An exploration

- The goal of this notebook is de-duplicate, validate, clean, transform the bronze tables

In [1]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions
spark = DatabricksSession.builder.getOrCreate()

# Policy Table

In [2]:
policy_sdf = spark.read.table("`commercial-insurance-underwriting-solution-catalog`.`bronze`.`policy`")

In [3]:
policy_sdf.dtypes

[('ID', 'bigint'),
 ('Date_start_contract', 'string'),
 ('Date_last_renewal', 'string'),
 ('Date_next_renewal', 'string'),
 ('Date_birth', 'string'),
 ('Date_driving_licence', 'string'),
 ('Distribution_channel', 'bigint'),
 ('Seniority', 'bigint'),
 ('Policies_in_force', 'bigint'),
 ('Max_policies', 'bigint'),
 ('Max_products', 'bigint'),
 ('Lapse', 'bigint'),
 ('Date_lapse', 'string'),
 ('Payment', 'bigint'),
 ('Premium', 'double'),
 ('Cost_claims_year', 'double'),
 ('N_claims_year', 'bigint'),
 ('N_claims_history', 'bigint'),
 ('R_Claims_history', 'double'),
 ('Type_risk', 'bigint'),
 ('Area', 'bigint'),
 ('Second_driver', 'bigint'),
 ('Year_matriculation', 'bigint'),
 ('Power', 'bigint'),
 ('Cylinder_capacity', 'bigint'),
 ('Value_vehicle', 'double'),
 ('N_doors', 'bigint'),
 ('Type_fuel', 'string'),
 ('Length', 'double'),
 ('Weight', 'bigint')]

## Convert all date columns from strings to date

In [4]:
date_columns = ['Date_start_contract', 'Date_last_renewal', 'Date_next_renewal', 'Date_birth', 'Date_driving_licence', 'Date_lapse']
for each_date in date_columns:
    policy_sdf = policy_sdf.withColumn(each_date, functions.to_date(functions.col(each_date), 'dd/MM/yyyy'))

In [23]:
na_tokens = ['NA', 'N/A', 'NULL', '']

for c, dtype in policy_sdf.dtypes:
  if dtype == 'string':
      policy_sdf = policy_sdf.withColumn(
          c, functions.when(functions.col(c).isin(na_tokens), None).otherwise(functions.col(c))
      )

## Checking and dealing with null values
- Checked for NA's that are not rendered as null that spark can deal with downstream. No column contained any

In [12]:
string_columns = [columns for columns, _ in policy_sdf.dtypes if _ == 'string']
na_invalid_values = ['NA', 'N/A', 'NULL', '', 'None', 'null']
policy_sdf.select([
  functions.count(functions.when(functions.col(c).isin(na_tokens), c)).alias(c) for c in string_cols
]).toPandas()

,Type_fuel
0,0


## Checking out Duplicates

In [14]:
total_rows = policy_sdf.count()
distinct_rows = policy_sdf.distinct().count()
duplicate_rows = total_rows - distinct_rows
print(duplicate_rows)

0


## Any other business rule applied?
- None

## Write to silver layer now

In [17]:
spark.sql("CREATE SCHEMA IF NOT EXISTS `commercial-insurance-underwriting-solution-catalog`.`silver`")
CATALOG = "`commercial-insurance-underwriting-solution-catalog`"
policy_sdf.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.policy")


# Claims Table

In [2]:
claims_sdf = spark.read.table("`commercial-insurance-underwriting-solution-catalog`.`bronze`.`claims`")

In [6]:
claims_sdf.dtypes

[('ID', 'bigint'),
 ('Cost_claims_year', 'double'),
 ('Cost_claims_by_type', 'double'),
 ('Claims_type', 'string')]

- Nothing much to do here. Now writing to silver layer as well

In [8]:
spark.sql("CREATE SCHEMA IF NOT EXISTS `commercial-insurance-underwriting-solution-catalog`.`silver`")
CATALOG = "`commercial-insurance-underwriting-solution-catalog`"
claims_sdf.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.claims")